In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
import optuna
import warnings

warnings.filterwarnings("ignore")

In [2]:
train = pd.read_csv('/kaggle/input/first-competition-exhibition/train.csv')
test  = pd.read_csv('/kaggle/input/first-competition-exhibition/test.csv')
test_ids = test['id']

train = train.drop(columns=['id', 'Row#'])
test  = test.drop(columns=['id', 'Row#'])

In [3]:
def create_features(df):
    data = df.copy()
    
    # Basic aggregations
    data['total_bees'] = data['honeybee'] + data['bumbles'] + data['andrena'] + data['osmia']
    data['bees_per_clone'] = data['total_bees'] / (data['clonesize'] + 1e-6)
    data['osmia_honeybee_inter'] = data['osmia'] * data['honeybee']
    
    # Temperature features
    data['temp_range'] = data['MaxOfUpperTRange'] - data['MinOfLowerTRange']
    data['avg_temp'] = (data['AverageOfUpperTRange'] + data['AverageOfLowerTRange']) / 2
    
    # Polynomial features
    for col in ['clonesize', 'osmia', 'honeybee', 'RainingDays', 'temp_range']:
        data[f'{col}_sq'] = data[col] ** 2
        data[f'{col}_cube'] = data[col] ** 3
        data[f'{col}_sqrt'] = np.sqrt(data[col] + 1e-6)
    
    # Fruit-seed interaction
    data['fruit_seed_ratio'] = data['fruitmass'] / (data['seeds'] + 1e-6)
    
    # Interaction features
    data['honeybee_avg_temp'] = data['honeybee'] * data['avg_temp']
    data['bumbles_rain'] = data['bumbles'] * data['RainingDays']
    data['osmia_honeybee_ratio'] = data['osmia'] / (data['honeybee'] + 1e-6)
    
    # Clustering feature
    cluster_cols = ['clonesize', 'total_bees', 'avg_temp', 'RainingDays']
    scaler = StandardScaler()
    scaled = scaler.fit_transform(data[cluster_cols])
    data['cluster'] = KMeans(n_clusters=10, random_state=42, n_init='auto').fit_predict(scaled)
    
    return data

In [4]:
train_fe = create_features(train)
test_fe = create_features(test)

y = train_fe['yield']
X = train_fe.drop('yield', axis=1)

In [5]:
train_fe

,clonesize,honeybee,bumbles,andrena,osmia,MaxOfUpperTRange,MinOfUpperTRange,AverageOfUpperTRange,MaxOfLowerTRange,MinOfLowerTRange,...,RainingDays_cube,RainingDays_sqrt,temp_range_sq,temp_range_cube,temp_range_sqrt,fruit_seed_ratio,honeybee_avg_temp,bumbles_rain,osmia_honeybee_ratio,cluster
0,12.5,0.25,0.25,0.25,0.75,86.0,52.0,71.9,62.0,30.0,...,4096.0,4.000000,3136.00,175616.000,7.483315,0.012620,15.3375,4.00,2.999988,4
1,25.0,0.50,0.38,0.38,0.50,86.0,52.0,71.9,62.0,30.0,...,39304.0,5.830952,3136.00,175616.000,7.483315,0.013120,30.6750,12.92,0.999998,1
2,12.5,0.25,0.25,0.38,0.63,86.0,52.0,71.9,62.0,30.0,...,13824.0,4.898980,3136.00,175616.000,7.483315,0.012286,15.3375,6.00,2.519990,4
3,12.5,0.25,0.25,0.50,0.50,69.7,42.1,58.2,50.2,24.3,...,13824.0,4.898980,2061.16,93576.664,6.737952,0.012735,12.4250,6.00,1.999992,3
4,25.0,0.50,0.25,0.63,0.63,86.0,52.0,71.9,62.0,30.0,...,13824.0,4.898980,3136.00,175616.000,7.483315,0.012601,30.6750,6.00,1.259997,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14995,12.5,0.25,0.25,0.50,0.75,69.7,42.1,58.2,50.2,24.3,...,1.0,1.000000,2061.16,93576.664,6.737952,0.012065,12.4250,0.25,2.999988,2
14996,37.5,0.25,0.25,0.25,0.25,86.0,52.0,71.9,62.0,30.0,...,4096.0,4.000000,3136.00,175616.000,7.483315,0.012767,15.3375,4.00,0.999996,6
14997,25.0,0.50,0.25,0.38,0.75,94.6,57.2,79.0,68.2,33.0,...,13824.0,4.898980,3794.56,233744.896,7.848567,0.012554,33.7250,6.00,1.499997,9
14998,25.0,0.50,0.25,0.63,0.75,77.4,46.8,64.7,55.8,27.0,...,39304.0,5.830952,2540.16,128024.064,7.099296,0.012979,27.6250,8.50,1.499997,7


In [6]:
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 800, 2000),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05, log=True),
        'max_depth': trial.suggest_int('max_depth', 4, 8),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 12),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'gamma': trial.suggest_float('gamma', 0, 5),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 5),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 5),
        'tree_method': 'hist',
        'random_state': 42,
        'eval_metric': 'mae'
    }
    
    kf = KFold(n_splits=10, shuffle=True, random_state=42)
    mae_scores = []
    
    for tr_idx, val_idx in kf.split(X):
        model = XGBRegressor(**params)
        model.fit(X.iloc[tr_idx], y.iloc[tr_idx],
                  eval_set=[(X.iloc[val_idx], y.iloc[val_idx])],
                  early_stopping_rounds=100,
                  verbose=False)
        preds = model.predict(X.iloc[val_idx])
        mae_scores.append(mean_absolute_error(y.iloc[val_idx], preds))
        
    return np.mean(mae_scores)

In [7]:
study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=50)
best_params = study.best_trial.params
print("Best XGB params:", best_params)
print("Best CV MAE:", study.best_value)

# --- Cross-Validation + Blending XGB + LGBM ---
kf = KFold(n_splits=15, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(test_fe))
mae_scores = []

[I 2025-12-23 11:53:40,225] A new study created in memory with name: no-name-346b4b69-9907-462a-84ec-71320b36335b
[I 2025-12-23 11:53:46,644] Trial 0 finished with value: 250.27153650572913 and parameters: {'n_estimators': 1249, 'learning_rate': 0.044635901521768134, 'max_depth': 7, 'min_child_weight': 8, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.662397808134481, 'gamma': 0.2904180608409973, 'reg_alpha': 4.330880728874676, 'reg_lambda': 3.005575058716044}. Best is trial 0 with value: 250.27153650572913.
[I 2025-12-23 11:54:20,971] Trial 1 finished with value: 250.8840781689531 and parameters: {'n_estimators': 1650, 'learning_rate': 0.005242693862597309, 'max_depth': 8, 'min_child_weight': 10, 'subsample': 0.6849356442713105, 'colsample_bytree': 0.6727299868828402, 'gamma': 0.9170225492671691, 'reg_alpha': 1.5212112147976886, 'reg_lambda': 2.6237821581611893}. Best is trial 0 with value: 250.27153650572913.
[I 2025-12-23 11:54:39,600] Trial 2 finished with value: 250.043438

Best XGB params: {'n_estimators': 1432, 'learning_rate': 0.01743367822111884, 'max_depth': 5, 'min_child_weight': 9, 'subsample': 0.8840140056195193, 'colsample_bytree': 0.8959343749237776, 'gamma': 2.3107011128410497, 'reg_alpha': 2.3023238331832383, 'reg_lambda': 4.612980464211761}
Best CV MAE: 248.2536689886797


In [8]:
for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
    print(f"Fold {fold+1}/15")
    
    # XGB Model
    xgb_model = XGBRegressor(**best_params, random_state=42, tree_method='hist')
    xgb_model.fit(X.iloc[tr_idx], y.iloc[tr_idx],
                  eval_set=[(X.iloc[val_idx], y.iloc[val_idx])],
                  early_stopping_rounds=100, verbose=False)
    
    # LGBM Model
    lgb_model = LGBMRegressor(
        n_estimators=best_params['n_estimators'],
        learning_rate=best_params['learning_rate'],
        max_depth=best_params['max_depth'],
        subsample=best_params['subsample'],
        colsample_bytree=best_params['colsample_bytree'],
        reg_alpha=best_params['reg_alpha'],
        reg_lambda=best_params['reg_lambda'],
        random_state=42
    )
    lgb_model.fit(X.iloc[tr_idx], y.iloc[tr_idx])
    
    # Blending predictions
    val_pred = 0.5*xgb_model.predict(X.iloc[val_idx]) + 0.5*lgb_model.predict(X.iloc[val_idx])
    test_pred = 0.5*xgb_model.predict(test_fe) + 0.5*lgb_model.predict(test_fe)
    
    oof_preds[val_idx] = val_pred
    test_preds += test_pred / kf.n_splits
    
    mae_scores.append(mean_absolute_error(y.iloc[val_idx], val_pred))

print(f"Mean OOF MAE: {np.mean(mae_scores):.4f} ± {np.std(mae_scores):.4f}")

Fold 1/15
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001493 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1355
[LightGBM] [Info] Number of data points in the train set: 14000, number of used features: 41
[LightGBM] [Info] Start training from score 6019.083531
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gai

In [9]:
submission = pd.DataFrame({'id': test_ids, 'yield': test_preds})
submission.to_csv('submission.csv', index=False)
print("Submission file created: submission.csv")

Submission file created: submission.csv
